# Multi-Model Fine-Tuning with LoRA
Each model is trained with its **own tokenizer** to avoid tokenizer/vocab mismatch errors.

In [1]:
!pip install transformers datasets peft accelerate sentence-transformers evaluate google-generativeai

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 33.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 7.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 37.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 33.9 MB/s  0:00:00
   ━━

In [2]:
!pip install --upgrade transformers

In [3]:
import torch
import random
import numpy as np
from datasets import load_dataset, concatenate_datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [4]:
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
dolly  = load_dataset("databricks/databricks-dolly-15k", split="train")

def format_alpaca(x):
    return {"instruction": x["instruction"], "output": x["output"]}

def format_dolly(x):
    return {"instruction": x["instruction"], "output": x["response"]}

alpaca = alpaca.map(format_alpaca, remove_columns=alpaca.column_names)
dolly  = dolly.map(format_dolly,  remove_columns=dolly.column_names)

data = concatenate_datasets([alpaca, dolly])
print(len(data))

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15011/15011 [00:00<00:00, 15756.18 examples/s]

67013


In [5]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    return text.strip()

data = data.map(lambda x: {
    "instruction": clean_text(x["instruction"]),
    "output":      clean_text(x["output"])
})

data = data.filter(lambda x: len(x["instruction"]) > 5 and len(x["output"]) > 5)

Filter: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 67013/67013 [00:00<00:00, 343411.74 examples/s]


In [6]:
# 80 / 10 / 10 split
split1 = data.train_test_split(test_size=0.2, seed=42)
train_data = split1["train"]
temp_data  = split1["test"]

split2 = temp_data.train_test_split(test_size=0.5, seed=42)
val_data  = split2["train"]
test_data = split2["test"]

print(len(train_data), len(val_data), len(test_data))

52482 6560 6561


In [7]:
# --- Transformations ---

def char_reverse(text):
    return text[::-1]

def word_reverse(text):
    return " ".join(text.split()[::-1])

def apply_transformation(data_list, type_="char"):
    new_data = []
    for d in data_list:
        if type_ == "char":
            new_output = char_reverse(d["output"])
        elif type_ == "word":
            new_output = word_reverse(d["output"])
        else:
            new_output = d["output"]
        new_data.append({"instruction": d["instruction"], "output": new_output})
    return new_data

def mix_data(clean, corrupted, ratio=0.5):
    clean     = list(clean)
    corrupted = list(corrupted)
    random.shuffle(clean)
    random.shuffle(corrupted)
    k     = int(len(clean) * ratio)
    mixed = clean[:k] + corrupted[:k]
    random.shuffle(mixed)
    return mixed

char_data  = apply_transformation(train_data, "char")
train_mixed = mix_data(train_data, char_data, ratio=0.5)

print("Clean size :", len(train_data))
print("Mixed size :", len(train_mixed))

Clean size : 52482
Mixed size : 52482


In [ ]:
# Gemini counterfactual (optional – requires valid API key)
import google.generativeai as genai

genai.configure(api_key="GOOGLE GEMINI API KEY GET FROM GOOGLE AI STUDIO")
model_gemini = genai.GenerativeModel("gemini-1.5-flash")

def generate_counterfactual(text):
    prompt = f"Give a wrong but plausible answer for: {text}"
    try:
        response = model_gemini.generate_content(prompt)
        return response.text
    except:
        return "Incorrect answer"

/tmp/ipykernel_862/3082260394.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# MODEL LIST
# Each entry is (model_name, lora_target_modules)
# target_modules differ per architecture – this is the key fix!
# ─────────────────────────────────────────────────────────────────────────────

model_configs = [
    # small
    ("facebook/opt-125m",              ["q_proj", "v_proj"]),
    ("facebook/opt-350m",              ["q_proj", "v_proj"]),
    ("EleutherAI/pythia-160m",         ["query_key_value"]),
    ("EleutherAI/pythia-410m",         ["query_key_value"]),
    ("distilgpt2",                     ["c_attn"]),
    ("gpt2",                           ["c_attn"]),
    ("sshleifer/tiny-gpt2",            ["c_attn"]),
    ("Qwen/Qwen2-0.5B",               ["q_proj", "v_proj"]),

    # medium
    ("facebook/opt-1.3b",             ["q_proj", "v_proj"]),
    ("EleutherAI/pythia-1b",          ["query_key_value"]),
    ("microsoft/phi-1_5",             ["q_proj", "v_proj"]),
    ("microsoft/phi-2",               ["q_proj", "v_proj"]),
    ("tiiuae/falcon-rw-1b",           ["query_key_value"]),
    ("Qwen/Qwen2-1.5B",              ["q_proj", "v_proj"]),
    ("TinyLlama/TinyLlama-1.1B-Chat-v1.0", ["q_proj", "v_proj"]),
]

print(f"Total models to train: {len(model_configs)}")

Total models to train: 15


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# ── Tokenise raw list with a given tokenizer ─────────────────────────────────
def tokenize_dataset(raw_list, tokenizer, max_length=256):
    """Tokenise a list of {instruction, output} dicts with the given tokenizer."""
    tokenized = []
    for example in raw_list:
        text   = f"Instruction: {example['instruction']}\nAnswer: {example['output']}"
        tokens = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_length
        )
        tokens["labels"] = tokens["input_ids"].copy()
        tokenized.append(tokens)

    ds = Dataset.from_list(tokenized)
    ds.set_format(type="torch")
    return ds


# ── LoRA helper ───────────────────────────────────────────────────────────────
def get_lora_model(model, target_modules):
    config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none"
    )
    return get_peft_model(model, config)


print("Helpers defined.")

Helpers defined.


In [12]:
import gc
from transformers import Trainer, TrainingArguments

results = []   # stores (model_name, trained_model, tokenizer)

for model_name, lora_targets in model_configs:
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")

    safe_name = model_name.replace("/", "_")

    # ── 1. Load tokenizer for THIS model ─────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Every causal-LM needs a pad token; use eos if missing
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ── 2. Tokenise data with this tokenizer ─────────────────────────────────
    print("  Tokenising train data...")
    train_dataset = tokenize_dataset(train_mixed, tokenizer)

    print("  Tokenising val data...")
    val_dataset = tokenize_dataset(list(val_data), tokenizer)

    # ── 3. Load model ─────────────────────────────────────────────────────────
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True
    ).to(device)

    # Resize embeddings in case pad_token was added
    model.resize_token_embeddings(len(tokenizer))

    # ── 4. Apply LoRA ─────────────────────────────────────────────────────────
    try:
        model = get_lora_model(model, lora_targets)
        model.print_trainable_parameters()
    except Exception as e:
        print(f"  LoRA failed ({e}), training without LoRA")

    # ── 5. Training arguments ─────────────────────────────────────────────────
    args = TrainingArguments(
        output_dir=f"./results/{safe_name}",
        per_device_train_batch_size=4,
        num_train_epochs=0.2,
        save_strategy="epoch",
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",          # suppress wandb / tensorboard
        dataloader_pin_memory=False,
    )

    # ── 6. Trainer ────────────────────────────────────────────────────────────
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset
    )

    trainer.train()

    # ── 7. Store results and free GPU memory ──────────────────────────────────
    results.append((model_name, model, tokenizer))

    # Move model to CPU to free VRAM before next iteration
    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  Done: {model_name}")

print("\nAll models trained!")


  Training: facebook/opt-125m
  Tokenising train data...
  Tokenising val data...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 34719.02it/s]
The tied weights mapping and config for this model specifies to tie model.decoder.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 294,912 || all params: 125,528,832 || trainable%: 0.2349


Step,Training Loss
50,7.325513
100,4.590004
150,4.252031
200,4.083114
250,4.014997
300,4.039009
350,3.938495
400,3.885041
450,3.877303
500,3.874617


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


...

A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-rw-1b:
- configuration_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


  Tokenising train data...
  Tokenising val data...


A new version of the following files was downloaded from https://huggingface.co/tiiuae/falcon-rw-1b:
- modeling_falcon.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 292/292 [00:00<00:00, 867.04it/s]


trainable params: 1,572,864 || all params: 1,416,028,160 || trainable%: 0.1111


AttributeError: 'FalconModel' object has no attribute 'get_head_mask'

In [13]:
# Inference helper – uses the model's own tokenizer
def generate(model, tokenizer, prompt):
    model.to(device)
    inputs  = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [15]:
!pip install nltk rouge_score
import evaluate
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

bleu      = evaluate.load("bleu")
rouge     = evaluate.load("rouge")
sim_model = SentenceTransformer("all-mpnet-base-v2")

def compute_similarity(pred, ref):
    emb1 = sim_model.encode([pred])
    emb2 = sim_model.encode([ref])
    return cosine_similarity(emb1, emb2)[0][0]

def compute_metrics(pred, ref):
    b = bleu.compute(predictions=[pred], references=[[ref]])["bleu"]
    r = rouge.compute(predictions=[pred], references=[ref])["rougeL"]
    return b, r

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.5 MB/s  0:00:00 eta 0:00:01
  DEPRECATION: Building 'rouge_score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge_score'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=4d3f4172f9782b3163464b098723be2b87794582a9c2f10650aa9c2e3278dff2
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rouge_score]


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3515.94it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
final_results = []

for model_name, model, tokenizer in results:
    print(f"\nEvaluating: {model_name}")

    model.to(device)

    sim_scores   = []
    bleu_scores  = []
    rouge_scores = []

    for d in list(test_data)[:100]:
        pred = generate(model, tokenizer, d["instruction"])
        ref  = d["output"]

        sim    = compute_similarity(pred, ref)
        bleu_s, rouge_s = compute_metrics(pred, ref)

        sim_scores.append(sim)
        bleu_scores.append(bleu_s)
        rouge_scores.append(rouge_s)

    final_results.append({
        "model":      model_name,
        "similarity": np.mean(sim_scores),
        "bleu":       np.mean(bleu_scores),
        "rouge":      np.mean(rouge_scores)
    })

    model.cpu()
    torch.cuda.empty_cache()
    gc.collect()


Evaluating: facebook/opt-125m

Evaluating: facebook/opt-350m

Evaluating: EleutherAI/pythia-160m

Evaluating: EleutherAI/pythia-410m

Evaluating: distilgpt2

Evaluating: gpt2

Evaluating: sshleifer/tiny-gpt2

Evaluating: Qwen/Qwen2-0.5B

Evaluating: facebook/opt-1.3b

Evaluating: EleutherAI/pythia-1b

Evaluating: microsoft/phi-1_5

Evaluating: microsoft/phi-2


In [17]:
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
for res in final_results:
    print("\n---------------------------")
    print("Model     :", res["model"])
    print("Similarity:", round(float(res["similarity"]), 4))
    print("BLEU      :", round(float(res["bleu"]),       4))
    print("ROUGE-L   :", round(float(res["rouge"]),      4))


FINAL RESULTS

---------------------------
Model     : facebook/opt-125m
Similarity: 0.5594
BLEU      : 0.0229
ROUGE-L   : 0.1189

---------------------------
Model     : facebook/opt-350m
Similarity: 0.5607
BLEU      : 0.0216
ROUGE-L   : 0.1241

---------------------------
Model     : EleutherAI/pythia-160m
Similarity: 0.5477
BLEU      : 0.0219
ROUGE-L   : 0.1047

---------------------------
Model     : EleutherAI/pythia-410m
Similarity: 0.575
BLEU      : 0.0239
ROUGE-L   : 0.1354

---------------------------
Model     : distilgpt2
Similarity: 0.5887
BLEU      : 0.0337
ROUGE-L   : 0.1685

---------------------------
Model     : gpt2
Similarity: 0.5617
BLEU      : 0.0293
ROUGE-L   : 0.1435

---------------------------
Model     : sshleifer/tiny-gpt2
Similarity: 0.5598
BLEU      : 0.0143
ROUGE-L   : 0.054

---------------------------
Model     : Qwen/Qwen2-0.5B
Similarity: 0.5823
BLEU      : 0.0419
ROUGE-L   : 0.1704

---------------------------
Model     : facebook/opt-1.3b
Similarity